# <font size="+22" color="red"> ===============Getting viral tracing maps============</font>

# The one from claude looks better, but mine is the correct singal density as deinfed on the online paltform

- i need to download the viral tracin gmaps that were inejcted in the dorsal strictum, in the caudate putamen.
- I extracted the experiments visaully, then claude made a rule around coordinates
- "AMBCA injection experiments targeting the dorsal caudoputamen were selected based on proximity to our injection site (AP +0.2, ML +2.0 relative to bregma).
Experiments with unique injection coordinates within 1mm of our target were included (n=4 experiments). Experiments were excluded if DV exceeded 3.0mm or ML exceeded 2.6mm, indicating placement outside the dorsal striatum. Projection density maps from selected experiments were averaged to generate a single structural connectivity vector."

- for each set of coordinates, i am going to download the maps and then average them together to get a single map of the dorsal striatum connectivity.
- then i will average all the 4 groups and see how it looks like

In [50]:
import numpy as np
import nibabel as nib
import os
import os

# Set the FSL output type to NIFTI_GZ (compressed NIfTI format)
os.environ['FSLOUTPUTTYPE'] = 'NIFTI_GZ'

# Also ensure FSLDIR is set if needed
os.environ['FSLDIR'] = '/usr/local/fsl'
os.environ['PATH'] += ':/usr/local/fsl/bin'
#from fsl.wrappers import fslmaths

In [51]:
PROCESSING_DIR = '/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps'

In [52]:
!mkdir -p {PROCESSING_DIR}
!cd {PROCESSING_DIR}
!rm {PROCESSING_DIR}/*/*.nii.gz

EXP_set_1 = [127140981, 278434443, 308395312, 100142580, 112458831, 301620241, 307910595]
EXP_set_2 = [127762867, 293366741]
EXP_set_3 = [158019342]
EXP_set_4 = [158916311]
# make a list of lists of EXP_IDs
EXP_IDs =  [EXP_set_1, EXP_set_2, EXP_set_3, EXP_set_4]
THRESHOLD = 0.00031622776
# THRESHOLD = 0.0316227766

rm: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps/*/*.nii.gz: No such file or directory


In [53]:
for i in range(len(EXP_IDs)):
    EXPIERMENT_set = EXP_IDs[i]
    print(f"Set {i+1}:")
    for j in range(len(EXPIERMENT_set)):
        EXPERIMENT_ID = EXPIERMENT_set[j]
        print(f"Experiment ID: {EXPERIMENT_ID}")


Set 1:
Experiment ID: 127140981
Experiment ID: 278434443
Experiment ID: 308395312
Experiment ID: 100142580
Experiment ID: 112458831
Experiment ID: 301620241
Experiment ID: 307910595
Set 2:
Experiment ID: 127762867
Experiment ID: 293366741
Set 3:
Experiment ID: 158019342
Set 4:
Experiment ID: 158916311


In [54]:

for i in range(len(EXP_IDs)):
    EXPIERMENT_set = EXP_IDs[i]
    for j in range(len(EXPIERMENT_set)):
        EXPERIMENT_ID = EXPIERMENT_set[j]


        !echo "Experiment ID: {EXPERIMENT_ID}"
        !rm {PROCESSING_DIR}/{EXPERIMENT_ID}.zip
        !wget  http://api.brain-map.org/grid_data/download/{EXPERIMENT_ID}?include=density -O {PROCESSING_DIR}/{EXPERIMENT_ID}.zip
        !rm -rf {PROCESSING_DIR}/{EXPERIMENT_ID}
        !unzip -d {PROCESSING_DIR}/{EXPERIMENT_ID} -o {PROCESSING_DIR}/{EXPERIMENT_ID}.zip


        # renaming mhd file is somehow probelmatic, keep the energy.mhd and convert it to nii
        !c3d {PROCESSING_DIR}/{EXPERIMENT_ID}/density.mhd -o {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}.nii.gz
        # augment
        !python ~/Dropbox/SCRIPTS/nibabel_augmentation.py  {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}.nii.gz  {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz  0.001

        # remove s mat
        !fslorient -setsformcode 0 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz

        # rotate
        !3drefit -deoblique -orient ASL {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz

        # set q mat to match the template
        !fslorient -setqform  0.000000 0.000000 0.100000 -5.695000 -0.100000 0.000000 0.000000 5.350000 0.000000 -0.100000 0.000000 5.220000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz

        !fslorient -setsformcode 0 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz

        !fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000  0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz



        # # normalize the image between 0 and 1 (no -need, the image is already nromalized`)
        img_100_raw = nib.load(f"{PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz")
        img_100_data = img_100_raw.get_fdata()
        affine_100 = img_100_raw.affine
        # img_100_data_norm = (img_100_data - img_100_data.min()) / (img_100_data.max() - img_100_data.min())
        # img_100_norm = nib.Nifti1Image(img_100_data_norm, affine_100)
        # nib.save(img_100_norm, f"{PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100_norm.nii.gz")

        !fslorient -setqformcode 1 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz
        !fslorient -setqform  0.000000 0.000000 0.100000 -5.695000 -0.100000 0.000000 0.000000 5.350000 0.000000 -0.100000 0.000000 5.220000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz
        !fslorient -setsformcode 0 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz
        !fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000  0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz
        # print that the image is saved
        print(f"Image saved at {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz")
        print("############################################################################################################")

        # calculate the log10 of the image => no need, the image is already normalized
        epsilon = 1e-10  # Small value to replace zero or negative values
        img_100_data[img_100_data <= 0] = epsilon
        img_100_data_log = np.log10(img_100_data)

        img_100_log = nib.Nifti1Image(img_100_data_log, affine_100)
        nib.save(img_100_log, f"{PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100_log.nii.gz")

        !fslorient -setqformcode 1 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100_log.nii.gz
        !fslorient -setqform  0.000000 0.000000 0.100000 -5.695000 -0.100000 0.000000 0.000000 5.350000 0.000000 -0.100000 0.000000 5.220000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100_log.nii.gz
        !fslorient -setsformcode 0 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100_log.nii.gz
        !fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000  0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100_log.nii.gz


        # set any value below 10^-3.5 to zero => does not apply there, that's not projection volume
        !fslmaths {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100_log.nii.gz -thr {THRESHOLD}  {PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100_log_thr.nii.gz

        # average the log thresholded images for each set of coordinates and save the average image
        if j == 0:
            avg_img_data = nib.load(f"{PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz").get_fdata()
        else:
            avg_img_data += nib.load(f"{PROCESSING_DIR}/{EXPERIMENT_ID}/{EXPERIMENT_ID}_100.nii.gz").get_fdata()
        if j == len(EXPIERMENT_set) - 1:
            avg_img_data /= len(EXPIERMENT_set)
            avg_img = nib.Nifti1Image(avg_img_data, affine_100)
            nib.save(avg_img, f"{PROCESSING_DIR}/set_{i+1}_avg.nii.gz")
            !fslorient -setqformcode 1 {PROCESSING_DIR}/set_{i+1}_avg.nii.gz
            !fslorient -setqform  0.000000 0.000000 0.10000 -5.695000 -0.10000 0.000000 0.000000 5.350000 0.000000 -0.10000 0.000000 5.220000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/set_{i+1}_avg.nii.gz
            !fslorient -setsformcode 0 {PROCESSING_DIR}/set_{i+1}_avg.nii.gz
            !fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000  0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/set_{i+1}_avg.nii.gz
            print(f"Average image for set {i+1} saved at {PROCESSING_DIR}/set_{i+1}_avg.nii.gz")

# now average the 4 average images and save the final average image
avg_img_data = None
for i in range(len(EXP_IDs)):
    avg_img_data_i = nib.load(f"{PROCESSING_DIR}/set_{i+1}_avg.nii.gz").get_fdata()
    if avg_img_data is None:
        avg_img_data = avg_img_data_i
    else:
        avg_img_data += avg_img_data_i
avg_img_data /= len(EXP_IDs)
avg_img = nib.Nifti1Image(avg_img_data, affine_100)
nib.save(avg_img, f"{PROCESSING_DIR}/final_avg.nii.gz")
!fslorient -setqformcode 1 {PROCESSING_DIR}/final_avg.nii.gz
!fslorient -setqform  0.000000 0.000000 0.100000 -5.695000 -0.100000 0.000000 0.000000 5.350000 0.000000 -0.100000 0.000000 5.220000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/final_avg.nii.gz
!fslorient -setsformcode 0 {PROCESSING_DIR}/final_avg.nii.gz
!fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000  0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 {PROCESSING_DIR}/final_avg.nii.gz
print(f"Final average image saved at {PROCESSING_DIR}/final_avg.nii.gz")


Experiment ID: 127140981
rm: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps/127140981.zip: No such file or directory
--2026-04-29 13:31:26--  http://api.brain-map.org/grid_data/download/127140981?include=density
Resolving api.brain-map.org (api.brain-map.org)... 184.33.67.253, 35.81.178.119
Connecting to api.brain-map.org (api.brain-map.org)|184.33.67.253|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/octet-stream]
Saving to: ‘/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps/127140981.zip’

/Users/aeed/Documen     [ <=>                ] 120.47K   632KB/s    in 0.2s    

2026-04-29 13:31:27 (632 KB/s) - ‘/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps/127140981.zip’ saved [123363]

Archive:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/

In [39]:
import numpy as np
import nibabel as nib
import os
import subprocess

# Set the FSL output type to NIFTI_GZ (compressed NIfTI format)
os.environ['FSLOUTPUTTYPE'] = 'NIFTI_GZ'
os.environ['FSLDIR'] = '/usr/local/fsl'
os.environ['PATH'] += ':/usr/local/fsl/bin'

PROCESSING_DIR = '/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude'
os.makedirs(PROCESSING_DIR, exist_ok=True)

# --- Experiment sets grouped by unique coordinates ---
# Set 1: AP=0.14, ML=1.75, DV=2.53 (closest to injection site)
# Set 2: AP=-0.70, ML=2.17, DV=1.92
# Set 3: AP=0.96, ML=1.90, DV=1.95
# Set 4: AP=0.98, ML=1.90, DV=2.10
EXP_set_1 = [127140981, 278434443, 308395312, 100142580, 112458831, 301620241, 307910595]
EXP_set_2 = [127762867, 293366741]
EXP_set_3 = [158019342]
EXP_set_4 = [158916311]

EXP_sets = [EXP_set_1, EXP_set_2, EXP_set_3, EXP_set_4]

THRESHOLD = 0.00031622776  # 10^-3.5

# qform matrix matching your template
QFORM = ("0.000000 0.000000 0.100000 -5.695000 "
         "-0.100000 0.000000 0.000000 5.350000 "
         "0.000000 -0.100000 0.000000 5.220000 "
         "0.000000 0.000000 0.000000 1.000000")

SFORM_ZERO = ("0.000000 0.000000 0.000000 0.000000 "
              "0.000000 0.000000 0.000000 0.000000 "
              "0.000000 0.000000 0.000000 0.000000 "
              "0.000000 0.000000 0.000000 1.000000")


def run(cmd):
    """Run a shell command and print it."""
    print(f"  >> {cmd}")
    subprocess.run(cmd, shell=True, check=True)


def set_orientation(filepath):
    """Apply your standard orientation fixes to a NIfTI file."""
    run(f"fslorient -setsformcode 0 {filepath}")
    run(f"3drefit -deoblique -orient ASL {filepath}")
    run(f"fslorient -setqformcode 1 {filepath}")
    run(f"fslorient -setqform {QFORM} {filepath}")
    run(f"fslorient -setsformcode 0 {filepath}")
    run(f"fslorient -setsform {SFORM_ZERO} {filepath}")


def download_and_process(exp_id, processing_dir):
    """Download, convert, orient, and return processed file path for one experiment."""
    exp_dir = os.path.join(processing_dir, str(exp_id))
    zip_path = os.path.join(processing_dir, f"{exp_id}.zip")
    raw_path = os.path.join(exp_dir, f"{exp_id}.nii.gz")
    out_path = os.path.join(exp_dir, f"{exp_id}_100.nii.gz")

    # Download
    run(f"wget -q http://api.brain-map.org/grid_data/download/{exp_id} -O {zip_path}")
    os.makedirs(exp_dir, exist_ok=True)
    run(f"unzip -o -d {exp_dir} {zip_path}")

    # Convert from mhd to nifti
    run(f"c3d {exp_dir}/energy.mhd -o {raw_path}")

    # Augment (your custom script)
    run(f"python ~/Dropbox/SCRIPTS/nibabel_augmentation.py {raw_path} {out_path} 0.001")

    # Set orientation to match template
    set_orientation(out_path)

    return out_path


def process_experiment_set(exp_set, set_index, processing_dir):
    """
    Download and process all experiments in a set,
    average them, and save the set mean.
    Returns the path to the averaged map.
    """
    print(f"\n{'='*80}")
    print(f"Processing Set {set_index + 1}: {exp_set}")
    print(f"{'='*80}")

    processed_data = []
    ref_affine = None

    for exp_id in exp_set:
        print(f"\n--- Experiment {exp_id} ---")
        out_path = download_and_process(exp_id, processing_dir)

        img = nib.load(out_path)
        data = img.get_fdata()
        processed_data.append(data)

        if ref_affine is None:
            ref_affine = img.affine

    # Average within this coordinate set
    set_mean = np.mean(processed_data, axis=0)

    # Save set average (raw, before log or thresholding)
    set_mean_path = os.path.join(processing_dir, f"set_{set_index + 1}_mean.nii.gz")
    nib.save(nib.Nifti1Image(set_mean, ref_affine), set_mean_path)
    set_orientation(set_mean_path)
    print(f"\nSet {set_index + 1} mean saved: {set_mean_path}")

    return set_mean, ref_affine


# ============================================================
# MAIN PIPELINE
# ============================================================

set_means = []
ref_affine = None

for i, exp_set in enumerate(EXP_sets):
    set_mean, affine = process_experiment_set(exp_set, i, PROCESSING_DIR)
    set_means.append(set_mean)
    if ref_affine is None:
        ref_affine = affine

# --- Grand mean across coordinate sets (equal weight per set) ---
grand_mean = np.mean(set_means, axis=0)
grand_mean_path = os.path.join(PROCESSING_DIR, "grand_mean.nii.gz")
nib.save(nib.Nifti1Image(grand_mean, ref_affine), grand_mean_path)
set_orientation(grand_mean_path)
print(f"\nGrand mean saved: {grand_mean_path}")

# --- Log10 transform of grand mean ---
epsilon = 1e-10
grand_mean_clean = grand_mean.copy()
grand_mean_clean[grand_mean_clean <= 0] = epsilon
grand_mean_log = np.log10(grand_mean_clean)

grand_mean_log_path = os.path.join(PROCESSING_DIR, "grand_mean_log.nii.gz")
nib.save(nib.Nifti1Image(grand_mean_log, ref_affine), grand_mean_log_path)
set_orientation(grand_mean_log_path)
print(f"Grand mean (log10) saved: {grand_mean_log_path}")

# --- Threshold the log map ---
run(f"fslmaths {grand_mean_log_path} -thr {THRESHOLD} "
    f"{os.path.join(PROCESSING_DIR, 'grand_mean_log_thr.nii.gz')}")
print(f"Grand mean (log10, thresholded) saved: {os.path.join(PROCESSING_DIR, 'grand_mean_log_thr.nii.gz')}")

print("\n" + "="*80)
print("PIPELINE COMPLETE")
print(f"  Set means:     set_1_mean.nii.gz ... set_4_mean.nii.gz")
print(f"  Grand mean:    grand_mean.nii.gz")
print(f"  Grand log:     grand_mean_log.nii.gz")
print(f"  Grand log thr: grand_mean_log_thr.nii.gz")
print("="*80)



Processing Set 1: [127140981, 278434443, 308395312, 100142580, 112458831, 301620241, 307910595]

--- Experiment 127140981 ---
  >> wget -q http://api.brain-map.org/grid_data/download/127140981 -O /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981.zip
  >> unzip -o -d /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981.zip
Archive:  /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981.zip
  inflating: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/data_set.xml  
  inflating: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/ene

++ 3drefit: AFNI version=AFNI_21.3.09 (Nov 26 2021) [64-bit]
++ Authored by: RW Cox
++ Processing AFNI dataset /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/127140981_100.nii.gz
 + changing orientation codes
 + deoblique
 + loading and re-writing dataset /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/127140981_100.nii.gz (/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/127140981_100.nii.gz in NIFTI storage)
++ 3drefit processed 1 datasets


  >> fslorient -setqformcode 1 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/127140981_100.nii.gz
  >> fslorient -setqform 0.000000 0.000000 0.100000 -5.695000 -0.100000 0.000000 0.000000 5.350000 0.000000 -0.100000 0.000000 5.220000 0.000000 0.000000 0.000000 1.000000 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/127140981_100.nii.gz
  >> fslorient -setsformcode 0 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/127140981_100.nii.gz
  >> fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/127140981/127140981_100.nii.gz

--- Experiment 278434443 ---
  >

++ 3drefit: AFNI version=AFNI_21.3.09 (Nov 26 2021) [64-bit]
++ Authored by: RW Cox
++ Processing AFNI dataset /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/278434443/278434443_100.nii.gz
 + changing orientation codes
 + deoblique
 + loading and re-writing dataset /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/278434443/278434443_100.nii.gz (/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/278434443/278434443_100.nii.gz in NIFTI storage)
++ 3drefit processed 1 datasets


  >> fslorient -setsformcode 0 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/278434443/278434443_100.nii.gz
  >> fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/278434443/278434443_100.nii.gz

--- Experiment 308395312 ---
  >> wget -q http://api.brain-map.org/grid_data/download/308395312 -O /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/308395312.zip
  >> unzip -o -d /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/308395312 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/308395312.zip
Archive:  /Users/aeed/Documents/W

...

  >> fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/set_4_mean.nii.gz

Set 4 mean saved: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/set_4_mean.nii.gz
  >> fslorient -setsformcode 0 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz
  >> 3drefit -deoblique -orient ASL /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz
  >> fslorient -setqformcode 1 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz
  >> fslorient -setqform 0.000000 0.000000 0.100000 -5.695000 -0.100000 0.

++ 3drefit: AFNI version=AFNI_21.3.09 (Nov 26 2021) [64-bit]
++ Authored by: RW Cox
** AFNI converts NIFTI_datatype=64 (FLOAT64) in file /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz to FLOAT32
     Warnings of this type will be muted for this session.
     Set AFNI_NIFTI_TYPE_WARN to YES to see them all, NO to see none.
*+ WARNING: NO spatial transform (neither qform nor sform), in NIfTI file '/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz'
++ Processing AFNI dataset /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz
 + changing orientation codes
 + deoblique
 + loading and re-writing dataset /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz (/Users/aeed/Documents/Work/M83

  >> fslorient -setsformcode 0 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz
  >> fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz

Grand mean saved: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean.nii.gz
  >> fslorient -setsformcode 0 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz
  >> 3drefit -deoblique -orient ASL /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz
  >> fslorient -setqformcode 1 /Users/aeed/Documents/Work/M83_clea

++ 3drefit: AFNI version=AFNI_21.3.09 (Nov 26 2021) [64-bit]
++ Authored by: RW Cox
** AFNI converts NIFTI_datatype=64 (FLOAT64) in file /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz to FLOAT32
     Warnings of this type will be muted for this session.
     Set AFNI_NIFTI_TYPE_WARN to YES to see them all, NO to see none.
*+ WARNING: NO spatial transform (neither qform nor sform), in NIfTI file '/Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz'
++ Processing AFNI dataset /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz
 + changing orientation codes
 + deoblique
 + loading and re-writing dataset /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz (/Users/aeed/Do

  >> fslorient -setsformcode 0 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz
  >> fslorient -setsform 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 1.000000 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz
Grand mean (log10) saved: /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz
  >> fslmaths /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log.nii.gz -thr 0.00031622776 /Users/aeed/Documents/Work/M83_clearing/Manuscript_analysis/struc_func_connectivity/viral_tracing_maps_claude/grand_mean_log_thr.nii.gz
Grand mean (log10, thresholded) saved: /Users/aeed/Documents/Work/M83_clearin